In [ ]:
# Ensure that local modules are reloaded when edited
%load_ext autoreload
%autoreload 2

# Ensure that plots are displayed
%matplotlib inline

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

https://machinelearningmastery.com/training-the-transformer-model/

In [ ]:
from time import time

In [ ]:
from utils.dataset import Dataset
from utils.enums import EncodingCategorical, EncodingNumerical

dataset_name='medium_debug_v2-0.15-4_1.json.gz'
prefix=True
pretrain_percentage=0.0
vector_size=20
window_size=4
categorical_encoding=EncodingCategorical.WORD_2_VEC_C
numerical_encoding=EncodingNumerical.MIN_MAX_SCALING
fs_save=None

dataset = Dataset(dataset_name, 
                      beta=0.005, 
                      prefix=prefix,
                      pretrain_percentage=pretrain_percentage,
                      vector_size=vector_size,
                      window_size=window_size, 
                      categorical_encoding=categorical_encoding,
                      numerical_encoding=numerical_encoding,
                      fs_save=fs_save)

# W2V Encoding

In [ ]:
from utils.embedding.w2v import ProcessWord2VecEncoder
import numpy as np

In [ ]:
dataset.case_lens[0:20]

In [ ]:
np.mean(dataset.case_lens)

In [ ]:
dataset.features[0].shape

In [ ]:
np.set_printoptions(linewidth=200)
dataset.features[2][0:20]

In [ ]:
np.set_printoptions(linewidth=300)
dataset.features[5][0:20]

In [ ]:
w2v_encoder = ProcessWord2VecEncoder(
            encoders=dataset.encoders,
            pretrain_percentage=dataset.pretrain_percentage,
            attribute_types=dataset.attribute_types,
            event_attribute_keys=dataset.event_log.event_attribute_keys,
            features=dataset._features,
            event_log=dataset.event_log,
            vector_size=dataset.vector_size,
            window=dataset.window_size,
            fs_save=dataset.fs_save) 

In [ ]:
w2v_features, numeric_features, numeric_feature_names, w2v_feature_names = w2v_encoder.encode_features(average=False, 
                                                                                                       match_numerical=True)

In [ ]:
# (num_attribute, num_cases, num_events, vector_size) 
print(w2v_features.shape)
transposed_w2v_features = np.transpose(w2v_features, (1, 2, 0, 3))
print(transposed_w2v_features.shape)

In [ ]:
print(numeric_features.shape)
transposed_numeric_features = np.transpose(numeric_features, (1, 2, 0, 3))
print(transposed_numeric_features.shape)

In [ ]:
dataset.case_lens[13]

In [ ]:
transposed_numeric_features[12,:,0]

In [ ]:
transposed_w2v_features[12,:,0]

In [ ]:
# Shapes of the input data
num_traces, num_events, num_numeric_features, vector_size = transposed_numeric_features.shape
_, _, num_w2v_features, _ = transposed_w2v_features.shape

# Initialize the merged array
merged_features = np.zeros((num_traces, num_events, num_numeric_features + num_w2v_features, vector_size))

# Keep track of the current indices for numeric and w2v features
numeric_index, w2v_index = 0, 0

# Iterate over dataset.attribute_keys to place each feature in the correct order
for i, key in enumerate(dataset.attribute_keys):
    # print(key)
    if key in numeric_feature_names:
        # print("Numeric feature shape:", transposed_numeric_features[:, :, numeric_index, :].shape)
        # print("Target shape:", merged_features[:, :, i, :].shape)
        # print(numeric_index)
        # Place numeric feature in the merged array
        merged_features[:, :, i, :] = transposed_numeric_features[:, :, numeric_index, :]
        numeric_index += 1
    elif key in w2v_feature_names:
        # print("W2V feature shape:", transposed_w2v_features[:, :, w2v_index, :].shape)
        # print("Target shape:", merged_features[:, :, i, :].shape)
        # print(w2v_index)
        # Place w2v feature in the merged array
        merged_features[:, :, i, :] = transposed_w2v_features[:, :, w2v_index, :]
        w2v_index += 1
    else:
        raise ValueError(f"Unexpected attribute key '{key}' not found in either feature list.")

# The merged array now has features interlaced according to dataset.attribute_keys order
print("Merged features shape:", merged_features.shape)

In [ ]:
dataset.case_lens[10:20]

In [ ]:
merged_features[12][4]

In [ ]:
zero_event = np.zeros((num_numeric_features + num_w2v_features, vector_size))

trainX = []
trainY = []
for case, length in zip(merged_features, dataset.case_lens):
    last_event_index = length - 1
    trainY.append(case[length - 1].copy())

    # Remove the target event from the training data
    case[length - 1] = zero_event
    trainX.append(case)

trainX = np.array(trainX)
trainY = np.array(trainY)

In [ ]:
trainX.shape

In [ ]:
trainY.shape

In [ ]:
trainY[4]

In [ ]:
trainX[4,0]

In [ ]:
dim0, dim1, dim2, dim3 = trainX.shape
trainX = np.reshape(trainX, (dim0, dim1, dim2 * dim3))#, order='C')
trainX = np.reshape(trainX, (dim0, dim1 * dim2 * dim3))#, order='C')
trainX.shape

In [ ]:
dim0, dim1, dim2 = trainY.shape
trainY = np.reshape(trainY, (dim0, dim1 * dim2))#, order='C')
trainY.shape

# Token Encoding

In [ ]:
features = np.array(dataset.features)
cases = np.transpose(features, (1, 2, 0))

In [ ]:
# Debug Prints
print(features.shape)
print(cases.shape)
print(len(dataset.case_lens))
print(dataset.attribute_types)
print(dataset.event_log.event_attribute_keys)

In [ ]:
# Categorical filter TODO make sure the transformer model can also handle numerical values via multitask learning
from utils.enums import AttributeType

# Create a boolean mask for categorical attributes
categorical_mask = np.array(dataset.attribute_types) == AttributeType.CATEGORICAL
print(categorical_mask)
num_categorical_features = np.sum(categorical_mask)

# Use this mask to filter out only the categorical attributes in the last axis
categorical_cases = cases[..., categorical_mask]
print(categorical_cases.shape)

In [ ]:
categorical_keys = []
for type, key in zip(dataset.attribute_types, dataset.event_log.event_attribute_keys):
    if type == AttributeType.CATEGORICAL:
        categorical_keys.append(key)
print(categorical_keys)

In [ ]:
print(dataset.case_lens)

In [ ]:
zero_event = np.zeros((num_categorical_features))
print(zero_event.shape)

trainX_categorical = []
trainY_categorical = []
for index, (case, case_length) in enumerate(zip(categorical_cases, dataset.case_lens)):
    last_event_index = case_length - 1
    trainY_categorical.append(case[last_event_index].copy())

    # Remove the target event from the training data
    case[last_event_index] = zero_event
    trainX_categorical.append(case)

trainX_categorical = np.array(trainX_categorical, dtype=np.int64)
trainY_categorical = np.array(trainY_categorical, dtype=np.int64)

print(trainX_categorical.shape)
print(trainY_categorical.shape)

In [ ]:
dim0, dim1, dim2 = trainX_categorical.shape
trainX_categorical = np.reshape(trainX_categorical, (dim0, dim1 * dim2))#, order='C')
trainX_categorical.shape

In [ ]:
print(trainX_categorical[12])

In [ ]:
trainX_categorical[12,0:5]

In [ ]:
trainX_categorical[12,5:]

In [ ]:
trainY_categorical[0:20]

In [ ]:
print(trainY_categorical[12])

In [ ]:
vocab_size = 0
for encoder in dataset.encoders.values():
    largest_index = encoder.largest_attribute()
    if largest_index > vocab_size:
        vocab_size = largest_index + 1

In [ ]:
vocab_size

# Online Process Transformer Implementation

In [ ]:
from keras.optimizers import Adam # type: ignore
from keras.metrics import Mean # type: ignore
from tensorflow import data, train, GradientTape, function # type: ignore

from novel.transformer.components.transformer import TransformerModel
from novel.transformer.components.utils import LRScheduler, loss_fcn, accuracy_fcn, likelihood_fcn

In [ ]:
# Define the model parameters

h = 8  # Number of self-attention heads
d_k = 64  # Dimensionality of the linearly projected queries and keys
d_v = 64  # Dimensionality of the linearly projected values
d_model = 512  # Dimensionality of model layers' outputs
d_ff = 2048  # Dimensionality of the inner fully connected layer
n = 6  # Number of layers in the encoder stack

# Define the training parameters
batch_size = 32
beta_1 = 0.9
beta_2 = 0.98
epsilon = 1e-9
dropout_rate = 0.1

# Instantiate an Adam optimizer
optimizer = Adam(LRScheduler(d_model), beta_1, beta_2, epsilon)

# Prepare the training and test splits of the dataset
# dataset = PrepareDataset()
# trainX, trainY, train_orig, enc_seq_length, dec_seq_length, enc_vocab_size, dec_vocab_size = dataset('english-german-both.pkl')
# print(f'X Length: {trainX.shape}')
enc_seq_length = trainX_categorical.shape[1]
dec_seq_length = trainY_categorical.shape[1]
# h = 10

print(enc_seq_length, "Encoding Sequence Length")
print(dec_seq_length, "Decoding Sequence Length")


# d_model = enc_seq_length
# d_ff = enc_seq_length

# Prepare the dataset batches
train_dataset = data.Dataset.from_tensor_slices((trainX_categorical, trainY_categorical))
train_dataset = train_dataset.batch(batch_size)

# Create model
training_model = TransformerModel(
    enc_seq_length, 
    dec_seq_length, 
    h, d_k, 
    d_v, 
    d_model, 
    d_ff, 
    n, 
    dropout_rate,
    enc_vocab_size=vocab_size,
    dec_vocab_size=vocab_size)

# Include metrics monitoring
train_loss = Mean(name='train_loss')
train_accuracy = Mean(name='train_accuracy')
train_likelihood = Mean(name='train_likelihood')

# Create a checkpoint object and manager to manage multiple checkpoints
ckpt = train.Checkpoint(model=training_model, optimizer=optimizer)
ckpt_manager = train.CheckpointManager(ckpt, "./checkpoints", max_to_keep=3)

In [ ]:
predictions = []
targets = []
losses = []

# Speeding up the training process
@function
def train_step(encoder_input, decoder_input, decoder_output):
    with GradientTape() as tape:
        prediction = training_model(encoder_input, decoder_input, training=True)

        loss = loss_fcn(decoder_output, prediction)
        accuracy = accuracy_fcn(decoder_output, prediction)
        likelihood = likelihood_fcn(decoder_output, prediction)

    gradients = tape.gradient(loss, training_model.trainable_weights)
    optimizer.apply_gradients(zip(gradients, training_model.trainable_weights))

    train_loss(loss)
    train_accuracy(accuracy)
    train_likelihood(likelihood)

    return loss, prediction, decoder_output

train_loss.reset_states()
train_accuracy.reset_states()
train_likelihood.reset_states()

print(f"\nStart of training in {len(train_dataset)} batches")

start_time = time()

for step, (train_batchX, train_batchY) in enumerate(train_dataset):
    encoder_input = train_batchX[:, num_categorical_features:] #1:] #Skip the start symbol
    decoder_input =  train_batchY[:, 0:num_categorical_features]    #train_batchY[:, :-1] 
    decoder_output = train_batchY[:, :]

    # print(encoder_input.shape, "Encoder Input")
    # print(decoder_input.shape, "Decoder Input")
    # print(decoder_output.shape, "Decoder Output")

    loss, prediction, target = train_step(encoder_input, decoder_input, decoder_output)

    losses.append(loss.numpy())
    predictions.append(prediction.numpy())
    targets.append(decoder_output.numpy())

    if step % 25 == 0:
        print(f'Step {step} Loss {train_loss.result():.4f} Accuracy {train_accuracy.result():.4f} Likelihood {train_likelihood.result():.4f}')

# Print epoch number and loss value at the end of every epoch
print("Training Loss %.4f, Training Accuracy %.4f, Training Likelihood %.4f" % (train_loss.result(), train_accuracy.result(), train_likelihood.result()))

print("Total time taken: %.2fs" % (time() - start_time))

# Output analysis

In [ ]:
len(predictions)

In [ ]:
predictions[0].shape

In [ ]:
targets[0].shape

In [ ]:
print(losses)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(losses, label='Training Loss')
plt.xlabel('Batches')
plt.ylabel('Loss Value')
plt.title('Training Loss Over Time')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
predictions = np.vstack(predictions)

In [ ]:
predictions[0,0]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_prediction_bar_chart(predictions, word_labels=None):
    """
    Plots a bar chart of predicted word likelihoods.

    Parameters:
        predictions (np.ndarray): Array of probabilities (likelihoods) between 0 and 1.
        word_labels (list, optional): List of labels corresponding to the predictions. 
                                       If None, default labels will be used.
    """
    # Validate the input
    if not isinstance(predictions, np.ndarray):
        raise ValueError("Predictions should be a numpy array.")
    
    predictions = np.maximum(predictions, 0)
    num_predictions = len(predictions)
    
    # Create default word labels if none are provided
    if word_labels is None:
        word_labels = [f'Word {i}' for i in range(num_predictions)]

    # Create a bar chart
    plt.figure(figsize=(10, 6))  # Set the figure size
    plt.bar(word_labels, predictions, color='skyblue')  # Create bar chart

    # Add titles and labels
    plt.title('Predicted Word Likelihoods')
    plt.xlabel('Words')
    plt.ylabel('Likelihood (Probability)')
    # plt.ylim(0, 1)  # Set the y-axis limit to 0 and 1

    plt.xticks([])

    # Show the probability values on top of the bars
    # for i, v in enumerate(predictions):
    #     plt.text(i, v + 0.02, f"{v:.2f}", ha='center', va='bottom')

    # Display the bar chart
    plt.show()

In [ ]:
plot_prediction_bar_chart(predictions[600,3])

In [ ]:
plot_prediction_bar_chart(predictions[1,3])

In [ ]:
trainY_categorical.shape

In [ ]:
predictions.shape

In [ ]:
trainY_categorical[20000,0]

In [ ]:
predictions[20000,0]

In [ ]:
max(predictions[20000,0])

In [ ]:
likelihood_errors = np.zeros(trainY_categorical.shape)
for i, (event_pred, event_true) in enumerate(zip(predictions, trainY_categorical)):
    for j, (attribute_pred, attribute_true) in enumerate(zip(event_pred, event_true)):
        likelihood_true = attribute_pred[attribute_true]
        likelihood_pred = 0
        for pred in attribute_pred:
            if pred > likelihood_true:
                likelihood_pred += pred
        likelihood_errors[i,j] = likelihood_pred
        
likelihood_errors = np.transpose(likelihood_errors, (1, 0))

In [ ]:
likelihood_errors.shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_lines(errors, keys, title='Likelihood Errors per attribute over time', xlabel='Case Index', ylabel='Likelihood Error'):
    plt.figure(figsize=(12, 6))

    for i in range(errors.shape[0]):
        plt.plot(errors[i], label=keys[i])

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
def rolling_average_np(errors, window_size=10):
    kernel = np.ones(window_size) / window_size
    rolling_avg = np.array([np.convolve(attribute, kernel, mode='valid') for attribute in errors])
    return rolling_avg

smoothed_likelihood_errors = rolling_average_np(likelihood_errors, window_size=100)
print(smoothed_likelihood_errors.shape)

In [ ]:
plot_lines(smoothed_likelihood_errors, categorical_keys)

In [ ]:
from utils.enums import Perspective
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def plot_losses(results, labels, perspective, level, bucket=None, zoom=[[11000,13000],[-0.05, 0.2]]):
    def scatter_plot(ax, results, labels):
        y_values = results
        x_values = np.arange(len(results))
        ax.scatter(x_values[labels == 0], y_values[labels == 0], c='grey', s=3, label='Normal Prefixes', zorder=1)
        ax.scatter(x_values[labels == 1], y_values[labels == 1], c='red', s=3, label='Anomalous Prefixes', zorder=2)
        ax.grid(True)

    fig, ax = plt.subplots(figsize=(15, 6))

    labels = labels[:, perspective]
    scatter_plot(ax, results, labels)
    
    perspective_name = Perspective.values()[perspective]

    bucket_string = ''
    if bucket is not None:
        bucket_string = f'with bucket size {str(bucket)}'
    plt.title(f'Error per Prefix on the {perspective_name} perspective at {level} level {bucket_string}')
    plt.xlabel('Prefix Index')
    plt.ylabel('Loss')
    
    if zoom:
        axins = inset_axes(ax, width="60%", height="60%", loc='upper right')

        scatter_plot(axins, results, labels)
        axins.set_xlim(zoom[0])
        axins.set_ylim(zoom[1])
        _,_ = ax.indicate_inset_zoom(axins, edgecolor="black", linewidth=3)

    plt.xlabel('Batch Index')
    plt.ylabel('Error')
    plt.legend(loc='upper right')
    plt.show()

In [ ]:
dataset.case_labels.shape

In [ ]:
dataset.event_labels.shape

In [ ]:
dataset.attr_labels.shape

In [ ]:
likelihood_errors.shape

In [ ]:
perspective_likelihood_errors = likelihood_errors.copy()

In [ ]:
perspective_likelihood_errors = np.transpose(perspective_likelihood_errors, (1, 0))

In [ ]:
perspective_likelihood_errors.shape

In [ ]:
order_likelihood_errors = perspective_likelihood_errors[:,0]

In [ ]:
plot_losses(
    results=order_likelihood_errors, 
    labels=dataset.case_labels, 
    perspective=Perspective.ORDER, level='case', bucket=None, zoom=None)     

In [ ]:
attribute_likelihood_errors = perspective_likelihood_errors[:,1:]

In [ ]:
attribute_likelihood_errors_summed = np.mean(attribute_likelihood_errors, axis=1)

In [ ]:
attribute_likelihood_errors.shape

In [ ]:
plot_losses(
    results=attribute_likelihood_errors_summed, 
    labels=dataset.case_labels, 
    perspective=Perspective.ATTRIBUTE, level='case', bucket=None, zoom=None)   

In [ ]:
from sklearn.metrics import precision_recall_curve

def perspective_f1_score(true, pred):
    precision, recall, thresholds = precision_recall_curve(true, pred)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    return np.max(f1_scores)

In [ ]:
order_f1 = perspective_f1_score(dataset.case_labels[:,Perspective.ORDER], order_likelihood_errors)
attribute_f1 = perspective_f1_score(dataset.case_labels[:,Perspective.ATTRIBUTE], attribute_likelihood_errors_summed)

In [ ]:
print(f"F1-Score Order Case Level: \t {order_f1}")
print(f"F1-Score Attribute Case Level: \t {attribute_f1}")